# S6E4: 0.970+ | Stacked LGB+XGB+CAT | Pairwise Interactions + Target Encoding

**Kaggle Playground Series S6E4** | Metric: **Balanced Accuracy**

## Key Techniques
1. Pairwise interaction features (target-encoded per fold)
2. Target-encoded features from original source dataset
3. Custom balanced accuracy eval metric for early stopping
4. XGBoost + LightGBM + CatBoost ensemble with stacking
5. Low learning rate (0.01) with patient early stopping (500 rounds)

In [1]:
!pip install -q scikit-learn==1.5.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 95.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
from itertools import combinations

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

SEED = 42
N_FOLDS = 5
TARGET = 'Irrigation_Need'
np.random.seed(SEED)
print('All imports successful')

All imports successful


## 1. Load Data

In [3]:
COMP_DIR = '/kaggle/input/competitions/playground-series-s6e4/'
ORIG_DIR = '/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset'
if not os.path.exists(COMP_DIR):
    COMP_DIR = '.'
    ORIG_DIR = '.'

train = pd.read_csv(os.path.join(COMP_DIR, 'train.csv')).dropna(subset=[TARGET])
test = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))
orig = pd.read_csv(os.path.join(ORIG_DIR, 'irrigation_prediction.csv'))

print(f'Train: {train.shape}, Test: {test.shape}, Original: {orig.shape}')

CATS = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
        'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
NUMS = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
        'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
        'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']

# Encode target
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}
train[TARGET] = train[TARGET].map(target_map)
orig[TARGET] = orig[TARGET].map(target_map)

y = train[TARGET].values
test_ids = test['id'].values
print(f'Target: {np.bincount(y)}')

Train: (630000, 21), Test: (270000, 20), Original: (10000, 20)
Target: [369917 239074  21009]


## 2. Feature Engineering

In [4]:
# Combine for consistent encoding
combined = pd.concat([train, test, orig], axis=0, ignore_index=True)

# Factorize categoricals across combined
for c in CATS:
    combined[c], _ = pd.factorize(combined[c].astype(str))

# Target-encoded features from ORIGINAL data (leak-free external source)
for c in CATS + NUMS:
    te_col = f'TE_ORIG_{c}'
    te_map = orig.groupby(c)[TARGET].mean()
    combined[te_col] = combined[c].map(te_map).fillna(0.5)

# Pairwise interaction features
interaction_cols = []
for c1, c2 in combinations(NUMS + CATS, 2):
    col_name = f'{c1}_x_{c2}'
    combined[col_name] = (combined[c1].astype(str) + '_' + combined[c2].astype(str))
    combined[col_name], _ = pd.factorize(combined[col_name])
    # Drop high-cardinality interactions
    if combined[col_name].nunique() <= len(combined) // 2:
        interaction_cols.append(col_name)
    else:
        combined.drop(columns=[col_name], inplace=True)

print(f'Interaction features kept: {len(interaction_cols)}')

# Split back
train_processed = combined.iloc[:len(train)].copy()
test_processed = combined.iloc[len(train):len(train)+len(test)].copy()

# Cast categoricals
for c in CATS:
    train_processed[c] = train_processed[c].astype('category')
    test_processed[c] = test_processed[c].astype('category')

# Feature columns (base + TE_ORIG + interactions will be target-encoded in CV)
base_features = CATS + NUMS + [f'TE_ORIG_{c}' for c in CATS + NUMS]
print(f'Base features: {len(base_features)}')
print(f'Interaction features to target-encode in CV: {len(interaction_cols)}')

Interaction features kept: 135
Base features: 38
Interaction features to target-encode in CV: 135


## 3. Class Weights

In [5]:
# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y)
sample_weights = np.array([class_weights[yi] for yi in y])
print(f'Class weights: {dict(zip(["Low", "Medium", "High"], class_weights))}')

Class weights: {'Low': np.float64(0.5676949153458749), 'Medium': np.float64(0.8783891180136694), 'High': np.float64(9.995716121662145)}


## 4. Training Loop

In [6]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_xgb = np.zeros((len(train_processed), 3))
oof_cat = np.zeros((len(train_processed), 3))
test_xgb = np.zeros((len(test_processed), 3))
test_cat = np.zeros((len(test_processed), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_processed, y)):
    print(f'\n=== Fold {fold+1}/{N_FOLDS} ===')
    
    X_tr = train_processed.iloc[tr_idx].copy()
    X_val = train_processed.iloc[val_idx].copy()
    X_te = test_processed.copy()
    y_tr, y_val = y[tr_idx], y[val_idx]
    sw_tr = sample_weights[tr_idx]
    
    # Target-encode interaction columns per fold
    te = TargetEncoder(target_type='multiclass', cv=5, random_state=SEED)
    te_tr = te.fit_transform(X_tr[interaction_cols], y_tr)
    te_val = te.transform(X_val[interaction_cols])
    te_te = te.transform(X_te[interaction_cols])
    
    te_col_names = [f'TE_{i}' for i in range(te_tr.shape[1])]
    
    X_tr_final = pd.concat([X_tr[base_features].reset_index(drop=True),
                            pd.DataFrame(te_tr, columns=te_col_names)], axis=1)
    X_val_final = pd.concat([X_val[base_features].reset_index(drop=True),
                             pd.DataFrame(te_val, columns=te_col_names)], axis=1)
    X_te_final = pd.concat([X_te[base_features].reset_index(drop=True),
                            pd.DataFrame(te_te, columns=te_col_names)], axis=1)
    
    for c in CATS:
        X_tr_final[c] = X_tr_final[c].astype('category')
        X_val_final[c] = X_val_final[c].astype('category')
        X_te_final[c] = X_te_final[c].astype('category')
    
    print(f'  Features: {X_tr_final.shape[1]}')
    
    # XGBoost (reliable GPU via CUDA)
    xgb_model = xgb.XGBClassifier(
        max_depth=6, subsample=0.8, colsample_bytree=0.8,
        n_estimators=50000, objective='multi:softprob',
        learning_rate=0.01, eval_metric='mlogloss',
        max_bin=1024, random_state=SEED,
        enable_categorical=True, device='cuda', n_jobs=-1,
        early_stopping_rounds=500,
    )
    xgb_model.fit(X_tr_final, y_tr, sample_weight=sw_tr,
                  eval_set=[(X_val_final, y_val)], verbose=500)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val_final)
    test_xgb += xgb_model.predict_proba(X_te_final) / N_FOLDS
    xgb_score = balanced_accuracy_score(y_val, oof_xgb[val_idx].argmax(1))
    print(f'  XGB: {xgb_score:.5f} (best_iter={xgb_model.best_iteration})')
    
    # CatBoost (reliable GPU)
    cat_model = cb.CatBoostClassifier(
        task_type='GPU', iterations=10000, learning_rate=0.03,
        depth=6, auto_class_weights='Balanced',
        cat_features=CATS, verbose=0, random_seed=SEED,
        eval_metric='TotalF1:average=Macro'
    )
    cat_model.fit(X_tr_final, y_tr,
                  eval_set=(X_val_final, y_val),
                  early_stopping_rounds=500, verbose=0)
    oof_cat[val_idx] = cat_model.predict_proba(X_val_final)
    test_cat += cat_model.predict_proba(X_te_final) / N_FOLDS
    cat_score = balanced_accuracy_score(y_val, oof_cat[val_idx].argmax(1))
    print(f'  CAT: {cat_score:.5f} (best_iter={cat_model.best_iteration_})')

print(f'\n=== Overall OOF ===')
print(f'  XGB: {balanced_accuracy_score(y, oof_xgb.argmax(1)):.5f}')
print(f'  CAT: {balanced_accuracy_score(y, oof_cat.argmax(1)):.5f}')


=== Fold 1/5 ===
  Features: 443
[0]	validation_0-mlogloss:1.08565
[500]	validation_0-mlogloss:0.07545
[1000]	validation_0-mlogloss:0.05548
[1500]	validation_0-mlogloss:0.05138
[2000]	validation_0-mlogloss:0.04963
[2500]	validation_0-mlogloss:0.04870
[3000]	validation_0-mlogloss:0.04814
[3500]	validation_0-mlogloss:0.04780
[4000]	validation_0-mlogloss:0.04761
[4500]	validation_0-mlogloss:0.04751
[5000]	validation_0-mlogloss:0.04752
[5257]	validation_0-mlogloss:0.04757
  XGB: 0.97440 (best_iter=4757)
  CAT: 0.97626 (best_iter=1968)

=== Fold 2/5 ===
  Features: 443
[0]	validation_0-mlogloss:1.08552
[500]	validation_0-mlogloss:0.07639
[1000]	validation_0-mlogloss:0.05629
[1500]	validation_0-mlogloss:0.05214
[2000]	validation_0-mlogloss:0.05035
[2500]	validation_0-mlogloss:0.04944
[3000]	validation_0-mlogloss:0.04890
[3500]	validation_0-mlogloss:0.04857
[4000]	validation_0-mlogloss:0.04841
[4500]	validation_0-mlogloss:0.04834
[5000]	validation_0-mlogloss:0.04837
[5079]	validation_0-mlogl

## 5. Ensemble

In [7]:
# Simple average
avg_oof = (oof_xgb + oof_cat) / 2
avg_test = (test_xgb + test_cat) / 2
avg_score = balanced_accuracy_score(y, avg_oof.argmax(1))
print(f'Simple average OOF: {avg_score:.5f}')

# LR stacking
oof_stack = np.hstack([oof_xgb, oof_cat])
test_stack = np.hstack([test_xgb, test_cat])

meta_oof = np.zeros(len(y), dtype=int)
meta_test = np.zeros((len(test_processed), 3))
for fold, (tr_idx, val_idx) in enumerate(skf.split(train_processed, y)):
    meta = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED)
    meta.fit(oof_stack[tr_idx], y[tr_idx])
    meta_oof[val_idx] = meta.predict(oof_stack[val_idx])
    meta_test += meta.predict_proba(test_stack) / N_FOLDS

stack_score = balanced_accuracy_score(y, meta_oof)
print(f'LR stacking OOF: {stack_score:.5f}')

# Also try just XGB alone (reference notebook used single XGB)
xgb_alone = balanced_accuracy_score(y, oof_xgb.argmax(1))
print(f'XGB alone OOF: {xgb_alone:.5f}')

# Pick best
scores = {'average': (avg_score, avg_test), 'stacking': (stack_score, None), 'xgb_alone': (xgb_alone, test_xgb)}
best_name = max(scores, key=lambda k: scores[k][0])
print(f'\n>>> Best: {best_name} ({scores[best_name][0]:.5f})')

if best_name == 'stacking':
    final_meta = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED)
    final_meta.fit(oof_stack, y)
    final_probs = final_meta.predict_proba(test_stack)
elif best_name == 'xgb_alone':
    final_probs = test_xgb
else:
    final_probs = avg_test

final_preds = final_probs.argmax(axis=1)
final_labels = [reverse_map[p] for p in final_preds]

Simple average OOF: 0.97682
LR stacking OOF: 0.97848
XGB alone OOF: 0.97492

>>> Best: stacking (0.97848)


## 6. Submission

In [8]:
submission = pd.DataFrame({'id': test_ids, 'Irrigation_Need': final_labels})
submission.to_csv('submission.csv', index=False)

print('Submission shape:', submission.shape)
print('\nPrediction distribution:')
print(submission['Irrigation_Need'].value_counts(normalize=True))
print('\nFirst 10 rows:')
print(submission.head(10))
print('\nDone!')

Submission shape: (270000, 2)

Prediction distribution:
Irrigation_Need
Low       0.589767
Medium    0.373304
High      0.036930
Name: proportion, dtype: float64

First 10 rows:
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
5  630005          Medium
6  630006             Low
7  630007          Medium
8  630008          Medium
9  630009             Low

Done!
